# ROME Case Study — Guardrailed Multi-Agent System

Planner → Executor → Reviewer on **LangGraph**, with a **CrewAI** executor, **PydanticAI** schemas, **Mem0** memory (SQLite + Chroma) and **Groq**.

Runs in Colab or Jupyter. Put your key in Colab *Secrets* as `GROQ_API_KEY`, or skip the live cells and use the `--offline` cells.

In [ ]:
!git clone https://github.com/debray2523/rome-guardrailed-mas.git
%cd rome-guardrailed-mas
!pip install -q -r requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except Exception:
    pass  # local Jupyter: use .env
print('GROQ key set:', bool(os.getenv('GROQ_API_KEY')))

## 1. Graph structure

In [ ]:
from mas.graph import MASDeps, build_graph
from mas.offline import StubExecutor, stub_planner, stub_reviewer
print(build_graph(MASDeps(stub_planner(), stub_reviewer(), StubExecutor())).get_graph().draw_mermaid())

## 2. Live run (async, as Jupyter already has an event loop)

In [ ]:
from mas.agents import CrewAIExecutor, build_planner, build_reviewer
from mas.graph import run_task
from mas.memory import MemoryStore

deps = MASDeps(build_planner(), build_reviewer(), CrewAIExecutor(), MemoryStore.local())
app = build_graph(deps)
state = await run_task(app, 'deb', 'Outline a 3-step plan to migrate a CRM database. I prefer answers in bullet points.')
print(*state['trace'], sep='\n')
state['final']

## 3. Guardrail demo — reviewer always rejects
Executor runs exactly 3 times; the 4th request trips `execution_count > 3` and the graph returns the safe fallback.

In [ ]:
deps_reject = MASDeps(build_planner(), stub_reviewer('reject'), CrewAIExecutor(), None)
state = await run_task(build_graph(deps_reject), 'deb', 'Summarise my preferences')
print(*state['trace'], sep='\n')
assert state['final'].status == 'terminated_loop_cap'
state['final']

## 4. Cross-session memory
Each `!python` line is a **separate process**, i.e. a new session. Session 2 recalls what session 1 stored from the SQLite/Chroma files in `data/`.

In [ ]:
!python -m mas.cli run --user deb "Remember that I prefer answers in bullet points and work in Kolkata time"
!python -m mas.cli run --user deb "What answer format do I prefer?"
!python -m mas.cli memories --user deb

## 5. Offline (no API key) — scripted models, same graph and storage

In [ ]:
!python -m mas.cli run --offline --user demo "I prefer bullet points"
!python -m mas.cli run --offline --user demo "What format do I prefer?"
!python -m mas.cli run --offline --user demo --force-reject "anything"
!python -m pytest -q